# Fine-tuning BERTimbau Nível 2 (área responsável)

Fine-tuning do modelo `neuralmind/bert-base-portuguese-cased` para classificação de tickets ITIL no nível 2 (área técnica responsável).

- **Input:** `dataset.csv` com colunas `descricao` (texto cru BERTimbau é *cased*) e `area_responsavel` (compatível com o antigo `nivel2`)
- **Output:** modelo treinado em `bert_nivel2/` (zip baixado ao final)
- **Classes (5):** Rede, Servidor, Aplicações, BD, Segurança
- **Desbalanceamento:** mitigado com `CrossEntropyLoss` ponderada por classe (espelha o `class_weight='balanced'` do SVM)
- **Avaliação:** Stratified 5-Fold com predições out-of-fold (métrica reportada no TCC)
- **Critério de aceite:** F1 macro ≥ 0.65


In [ ]:
!pip install -q transformers==4.57.6 datasets==5.0.0 scikit-learn==1.9.0 accelerate==1.14.0

In [ ]:
from pathlib import Path

import pandas as pd

# Roda tanto localmente (qualquer cwd dentro do repo) quanto no Colab (upload).
CANDIDATOS = [
    Path("data/dataset_nivel_2.csv"),
    Path("../data/dataset_nivel_2.csv"),
    Path(__file__).resolve().parent.parent / "data/dataset_nivel_2.csv"
    if "__file__" in dir()
    else Path("data/dataset_nivel_2.csv"),
]
local = next((c for c in CANDIDATOS if c.exists()), None)

if local is not None:
    df = pd.read_csv(local)
    print(f"Lido localmente: {local}")
else:
    from google.colab import files

    uploaded = files.upload()  # selecione o dataset_nivel_2.csv
    df = pd.read_csv(list(uploaded.keys())[0])

# Compatibilidade: aceita tanto 'area_responsavel' (Nível 2) quanto o antigo 'nivel2'
if "area_responsavel" not in df.columns and "nivel2" in df.columns:
    df = df.rename(columns={"nivel2": "area_responsavel"})

print(f"Linhas: {len(df)}")
print(df.head())
print(df["area_responsavel"].value_counts())

In [ ]:
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

TARGET_COL = "area_responsavel"
TEXT_COL = "descricao"

# Filtra linhas com label nulo
df = df.dropna(subset=[TARGET_COL, TEXT_COL]).reset_index(drop=True)

le = LabelEncoder()
df["label"] = le.fit_transform(df[TARGET_COL])
num_labels = len(le.classes_)
id2label = {i: c for i, c in enumerate(le.classes_)}
label2id = {c: i for i, c in id2label.items()}

print(f"Classes ({num_labels}): {list(le.classes_)}")

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=RANDOM_SEED
)

train_ds = Dataset.from_pandas(
    train_df[[TEXT_COL, "label"]].rename(columns={TEXT_COL: "text"})
)
val_ds = Dataset.from_pandas(
    val_df[[TEXT_COL, "label"]].rename(columns={TEXT_COL: "text"})
)

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_LENGTH = 192

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(
        batch["text"], truncation=True, padding=True, max_length=MAX_LENGTH
    )


train_tok = train_ds.map(tokenize, batched=True)
val_tok = val_ds.map(tokenize, batched=True)

In [ ]:
import json

import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

# ─── Pesos por classe (espelha o class_weight='balanced' do SVM) ─────────────
# Calculados APENAS sobre o conjunto de treino, para não vazar informação do val.
# Fórmula sklearn 'balanced': peso[c] = n_amostras / (n_classes * contagem[c]).
class_weights = torch.tensor(
    compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_labels),
        y=train_df["label"].values,
    ),
    dtype=torch.float,
)
print("Pesos por classe:")
for i, w in enumerate(class_weights.tolist()):
    print(f"  {id2label[i]:12} {w:.3f}")


class WeightedLossTrainer(Trainer):
    """Trainer com CrossEntropyLoss ponderada por classe (mitiga desbalanceamento)."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = (
            self._class_weights.to(logits.device)
            if self._class_weights is not None
            else None
        )
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1_macro,
    }


OUTPUT_DIR = "bert_nivel2"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=20,
    seed=RANDOM_SEED,
    report_to="none",
)

trainer = WeightedLossTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

In [ ]:
trainer.train()

eval_metrics = trainer.evaluate()
print("Metrics finais:", eval_metrics)

# Classification report detalhado
from sklearn.metrics import classification_report  # noqa: E402

preds = trainer.predict(val_tok)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = val_tok["label"]

report = classification_report(
    y_true, y_pred, target_names=list(le.classes_), output_dict=True, zero_division=0
)
print(
    classification_report(
        y_true, y_pred, target_names=list(le.classes_), zero_division=0
    )
)

with open(f"{OUTPUT_DIR}/training_metrics.json", "w") as f:
    json.dump(
        {"eval": eval_metrics, "classification_report": report},
        f,
        indent=2,
        ensure_ascii=False,
    )

## Avaliação robusta Stratified K-Fold (out-of-fold)

O split único de 20% deixa as classes minoritárias com 3–6 exemplos na validação, tornando o F1 por classe instável (BD oscila entre 0.50 e 0.80 dependendo da semente). Com **5-fold CV** cada ticket é predito exatamente uma vez (quando cai no fold de teste), gerando predições *out-of-fold* (OOF) para os **185 tickets**. O `classification_report` agregado usa então o **support real** (ex.: BD = 13, não 3), e reportamos **média ± desvio** do F1 macro.

> Esta é a métrica que deve ir para o TCC. O modelo salvo (células seguintes) continua sendo o treinado no split único, para deploy via `CascadeClassifier`.


In [ ]:
import gc

from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5
EPOCHS_CV = 6  # fixo (definido em experimento preliminar) SEM seleção no fold de teste

X = df[TEXT_COL].values
y = df["label"].values

oof_pred = np.full(len(df), -1, dtype=int)
fold_f1 = []

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
    tr = df.iloc[tr_idx]
    te = df.iloc[te_idx]

    tr_ds = Dataset.from_pandas(
        tr[[TEXT_COL, "label"]].rename(columns={TEXT_COL: "text"}),
        preserve_index=False,
    ).map(tokenize, batched=True)
    te_ds = Dataset.from_pandas(
        te[[TEXT_COL, "label"]].rename(columns={TEXT_COL: "text"}),
        preserve_index=False,
    ).map(tokenize, batched=True)

    # pesos 'balanced' calculados SÓ no treino do fold (sem vazamento)
    w = torch.tensor(
        compute_class_weight(
            class_weight="balanced",
            classes=np.arange(num_labels),
            y=tr["label"].values,
        ),
        dtype=torch.float,
    )

    fold_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
    )

    fold_args = TrainingArguments(
        output_dir=f"cv_fold{fold}",
        num_train_epochs=EPOCHS_CV,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="no",
        save_strategy="no",
        logging_strategy="no",
        seed=RANDOM_SEED,
        report_to="none",
    )

    fold_trainer = WeightedLossTrainer(
        model=fold_model,
        args=fold_args,
        train_dataset=tr_ds,
        tokenizer=tokenizer,
        class_weights=w,
    )
    fold_trainer.train()

    preds = np.argmax(fold_trainer.predict(te_ds).predictions, axis=-1)
    oof_pred[te_idx] = preds

    f1 = f1_score(te["label"].values, preds, average="macro", zero_division=0)
    fold_f1.append(f1)
    print(f"Fold {fold}: F1 macro = {f1:.3f}")

    # libera a GPU entre folds
    del fold_model, fold_trainer
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nF1 macro por fold: {[round(f, 3) for f in fold_f1]}")
print(f"F1 macro CV (5-fold): {np.mean(fold_f1):.3f} +/- {np.std(fold_f1):.3f}")

print("\n=== Classification report agregado (OOF todos os 185 tickets) ===")
print(
    classification_report(y, oof_pred, target_names=list(le.classes_), zero_division=0)
)

print("=== Matriz de confusao (OOF) ===")
print("Classes:", list(le.classes_))
print(confusion_matrix(y, oof_pred))

# salva resultados do CV
cv_results = {
    "n_folds": N_FOLDS,
    "epochs_cv": EPOCHS_CV,
    "f1_macro_per_fold": [round(f, 4) for f in fold_f1],
    "f1_macro_mean": round(float(np.mean(fold_f1)), 4),
    "f1_macro_std": round(float(np.std(fold_f1)), 4),
    "oof_classification_report": classification_report(
        y, oof_pred, target_names=list(le.classes_), output_dict=True, zero_division=0
    ),
}
with open("bert_nivel2_cv_results.json", "w") as f:
    json.dump(cv_results, f, indent=2, ensure_ascii=False)
print("\nResultados do CV salvos em bert_nivel2_cv_results.json")

In [ ]:
import matplotlib.pyplot as plt

CLASSES = list(le.classes_)
cm = confusion_matrix(y, oof_pred, labels=range(num_labels))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(len(CLASSES)))
ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
ax.set_title("BERT Matriz de confusão (OOF, CV 5-fold, Nível 2)")

vmax = cm.max()
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        value = cm[i, j]
        color = "white" if value > vmax / 2 else "black"
        ax.text(j, i, str(value), ha="center", va="center", color=color)

fig.colorbar(im, ax=ax, label="Contagem (OOF)")
fig.tight_layout()
plt.show()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

!zip -r {OUTPUT_DIR}.zip {OUTPUT_DIR}/

from google.colab import files  # noqa: E402

files.download(f"{OUTPUT_DIR}.zip")

## Próximos passos

1. Extrair o zip baixado em `backend/models/bert/nivel2/` na raiz do repo
2. O CascadeClassifier vai detectar automaticamente o modelo via `is_trained()`
3. Critério de aceite: F1 macro ≥ 0.65